# **on-premise LLM server and client**

In [1]:
!echo "Installing Ollama..."
# Use the official install script
!curl -fsSL https://ollama.com/install.sh | sh
print("✅ Ollama is installed.")

Installing Ollama...
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
✅ Ollama is installed.


In [2]:
import os
import time

pid = !pgrep ollama
if pid:
    print("Ollama server is already running.")
else:
    print("Starting Ollama server...")
    os.system("nohup ollama serve > ollama.log 2>&1 &")
    time.sleep(5)
    print("✅ Ollama server started in the background.")

Starting Ollama server...
✅ Ollama server started in the background.


In [3]:
print("Downloading model 'gemma:2b'. This may take a minute...")
# This command talks to the server we just started
!ollama pull gemma:2b
print("✅ Model downloaded.")


✅ Model downloaded.


In [ ]:
!pip install -U langchain-ollama langchain-core pydantic
print("✅ Installed 'langchain-ollama' and other libraries.")

In [5]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import Literal

print("Setting up LangChain client...")

# schema
class TextClassifier(BaseModel):
    category: Literal["Sales", "Support", "Billing", "Feedback"] = Field(
        description="The category that best fits the text."
    )
    sentiment: Literal["Positive", "Neutral", "Negative"] = Field(
        description="The sentiment of the text."
    )

# model
llm = ChatOllama(model="gemma:2b", temperature=0)

# Bind the schema to it
structured_llm = llm.with_structured_output(TextClassifier)

# prompt
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a text classifier. Classify the text based on the schema."),
        ("human", "Please classify the following text: {text_input}")
    ]
)

# the chain
classifier_chain = prompt | structured_llm

print("✅ Classifier chain created. Running a test...")

# run test
new_text = "I'm having trouble logging into my account, the password reset isn't working."
result = classifier_chain.invoke({"text_input": new_text})

print("\n--- TEST COMPLETE ---")
print(f"Input: '{new_text}'")
print(f"Output: {result}")

Setting up LangChain client...
✅ Classifier chain created. Running a test...

--- TEST COMPLETE ---
Input: 'I'm having trouble logging into my account, the password reset isn't working.'
Output: category='Support' sentiment='Negative'
